# CrocoDash Tutorial with GridSketcher (experimental)

```{admonition} Experimental
:class: warning

This notebook uses `GridSketcher`, an interactive grid tool on a development branch of
`mom6_forge` (`gridsketcher` on github.com/amrhein/mom6_forge). It is a preliminary tool:
expect rough edges, and please report problems.
```

This is the [Getting Started](../tutorial-ocn.ipynb) tutorial with two changes: the
horizontal grid is sketched on a map with `GridSketcher`, which builds an orthogonal grid
that can follow the coast, and the open boundaries come from the final wet mask after the
Topo Editor.

A typical workflow of utilizing CrocoDash consists of four main steps:

1. Generate a regional MOM6 domain.
2. Create the CESM case.
3. Prepare forcing data.
4. Build and run the case.

## Setup

**One time, on NCAR JupyterHub:** the sketch needs the browser half of ipympl
(`jupyter-matplotlib`), which this JupyterHub environment doesn't ship. The cell
below copies it in automatically if needed; if it does, restart your Jupyter
server once (File > Hub Control Panel > Stop My Server, then start it again) and
re-open this notebook.

The cell after that clones the `gridsketcher` branch of `mom6_forge` (the first
time) and puts it ahead of the installed `mom6_forge`. Run both cells first,
before any CrocoDash import, after a kernel restart.

In [ ]:
import json
import shutil
import sys
from pathlib import Path

# The widget's browser half (jupyter-matplotlib) must match this kernel's ipympl
LABEXT = Path.home() / ".local/share/jupyter/labextensions/jupyter-matplotlib"
src = Path(sys.prefix) / "share/jupyter/labextensions/jupyter-matplotlib"


def version(path):
    return json.loads((path / "package.json").read_text())["version"]


if not src.exists():
    print("No jupyter-matplotlib labextension in", src, "- the widget may show blank.")
elif LABEXT.exists() and version(LABEXT).split(".")[:2] == version(src).split(".")[:2]:
    print("jupyter-matplotlib", version(LABEXT), "is installed. If the widget below says")
    print("jupyter-matplotlib is not registered, restart your Jupyter server once.")
else:
    if LABEXT.exists():
        old = LABEXT.with_name(f"jupyter-matplotlib-{version(LABEXT)}.bak")
        shutil.rmtree(old, ignore_errors=True)
        LABEXT.rename(old)
        print("Moved the mismatched copy to", old)
    shutil.copytree(src, LABEXT)
    print("Installed jupyter-matplotlib", version(src), "into", LABEXT.parent)
    print("Now restart your Jupyter server once (File > Hub Control Panel > Stop My Server,")
    print("then start it again), re-open this notebook and run it from the top.")


In [ ]:
import subprocess
import sys
from pathlib import Path

MOM6_FORGE = Path("/glade/work/damrhein/gridsketcher_workshop")
if not MOM6_FORGE.exists():
    url = "https://github.com/amrhein/mom6_forge.git"
    subprocess.run(["git", "clone", "-b", "gridsketcher", url, MOM6_FORGE], check=True)
sys.path.insert(0, str(MOM6_FORGE))

import cartopy

# Natural Earth land and coastlines staged on GLADE, so nothing is downloaded
cartopy.config["pre_existing_data_dir"] = "/glade/work/damrhein/gridsketcher_workshop/cartopy"
import mom6_forge

print("mom6_forge from", mom6_forge.__file__)

%matplotlib ipympl

# SECTION 1: Generate a regional MOM6 domain

We begin by defining a regional MOM6 domain using CrocoDash. To do so, we first generate a horizontal grid. We then generate the topography by remapping an existing bathymetric dataset to our horizontal grid. Finally, we define a vertical grid.

## Step 1.1: Horizontal Grid

The bathymetry file is used twice: for the faint depth contours in the sketch and for the
topography in Step 1.2. Choose it by grid resolution:

- **Cells of about 2 km or more:** GEBCO at 1 arc-minute (`coarse_x4`, ~1.9 km), below.
- **Finer cells:** the full 15" GEBCO (`/glade/campaign/cgd/oce/projects/CROCODILE/workshops/2026/CrocoDash/data/gebco_2026/GEBCO_2026.nc`, 7.5 GB). Cut a region wider than
  the grid by more than 0.5° first, so that remapping reads only what it needs.

`MIN_DEPTH` turns shallower cells into land: the tutorial uses 9.5 m; a shallow estuary
may need about 2 m.

📖 [Grid documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/1_grids.html)

In [ ]:
BATHY = Path("/glade/campaign/cgd/oce/projects/CROCODILE/workshops/2026/CrocoDash/data/gebco_2026/GEBCO_2026_coarse_x8.nc").with_name("GEBCO_2026_coarse_x4.nc")
if not BATHY.exists():  # ~3.7 km, where the x4 file is missing
    BATHY = Path("/glade/campaign/cgd/oce/projects/CROCODILE/workshops/2026/CrocoDash/data/gebco_2026/GEBCO_2026_coarse_x8.nc")
MIN_DEPTH = 9.5  # in meters
# Time-mean GLORYS 1/12° sea surface height (m), drawn as faint contours
SSH = Path("/glade/work/damrhein/gridsketcher_data/glorys_mean_ssh_1_12.nc")

Edit the outline on the map: drag a corner or point to move it, press an edge to add a
point, right-click a point to delete it, double-click a point to make it a corner or not.
Drag on empty map to pan, scroll to zoom, **Clear all** to draw a new outline. Set
**Resolution (km)**, check the side shading (pink = open sea, ✕ = a short or coast-hugging
open stretch), then press **Build final grid**.

The faint coloured contours are time-mean GLORYS sea surface height, shown as guidance for placing open boundaries.
**Depth contours** and **Mean SSH** toggle the two overlays, and each one's slider sets its
number of contours.

In [ ]:
from mom6_forge.grid_sketcher import GridSketcher

sketch = GridSketcher(
    name="gs_california", bathymetry=BATHY, ssh=SSH
)  # opens on California
sketch

In [ ]:
grid = sketch.grid
if grid is None:
    raise RuntimeError("Press Build final grid above, then run this cell again.")
print(grid.nx, "x", grid.ny, "cells; open sides in the sketch:", sketch.open_boundaries)

## Step 1.2: Topography


In [ ]:
from CrocoDash.topo import Topo

topo = Topo(
    grid = grid,
    min_depth = MIN_DEPTH,
)
topo.set_from_dataset(
    bathymetry_path = BATHY,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
    depth_method = "xesmf",
    mask_method = "dataset"
)
topo.keep_largest_basin()  # drop lakes and cut-off bays

Edit depths and the land mask in the Topo Editor. It also shows each side's open
stretches in magenta, with an ✕ on short or coast-hugging ones.

In [ ]:
from CrocoDash.topo_editor import TopoEditor

TopoEditor(topo, open_boundaries=True)

## Step 1.3: Open Boundaries

This cell checks the final wet mask and sets `boundaries`, the sides that get ocean
forcing in Section 3. Re-run it after editing the mask.

In [ ]:
from mom6_forge import corner_diagnostics

runs = corner_diagnostics.open_boundary_runs(grid, ocean=topo.tmask.values.astype(bool))
sides = corner_diagnostics.OBC_SIDES
tiny = [(side, r) for side in sides for r in runs[side]["runs"] if r["tiny"]]
if tiny:  # worded as the TopoEditor has it
    n = len(tiny)
    print(f"Warning: {n} small open boundar{'y' if n == 1 else 'ies'} along coastline")
    for side, r in tiny:
        lon, k = (r["lon"] + 180) % 360 - 180, r["length_cells"]
        print(f"  {side}: {k} cell{'s' * (k > 1)}, lat {r['lat']:.2f} lon {lon:.2f}")
    print("  Mask them in the editor above and re-run this cell, or move that side.")
for side in sides:
    if runs[side]["alternating"]:
        n = runs[side]["n_transitions"]
        print(f"Warning: {side} side runs along the coast ({n} land/sea changes).")

boundaries = [side for side in sides if runs[side]["runs"]]
if boundaries != sketch.open_boundaries:
    print(f"Note: the sketch showed {sketch.open_boundaries}; the topography decides.")
cells = [f"{s} {runs[s]['ocean'].sum()}/{runs[s]['ocean'].size}" for s in boundaries]
print("Open boundaries (open/all cells):", ", ".join(cells) or "none")

```{admonition} Going back to the sketch
:class: tip

If the Topo Editor or the check above shows a problem (a sliver of open boundary, a side
along the coast, a bay you want in or out), go back to the sketch, edit the outline, press
**Build final grid** again and re-run every cell from Step 1.1 down. Mask edits in the
Topo Editor belong to one grid and are lost on a rebuild, so settle the outline first and
edit the mask last.
```

## Step 1.4: Vertical Grid

📖 [Grid documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/1_grids.html)

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid  = VGrid.hyperbolic(
    nk = 32, # number of vertical levels
    depth = topo.max_depth,
    ratio=50.0 # target ratio of top to bottom layer thicknesses
)

In [ ]:
import matplotlib.pyplot as plt
plt.close()
# Create the plot
for depth in vgrid.zi:
    plt.axhline(y=depth, linestyle='-')  # Horizontal lines

plt.ylim(max(vgrid.zi) + 10, min(vgrid.zi) - 10)  # Invert y-axis so deeper values go down
plt.ylabel("Depth")
plt.title("Vertical Grid")
plt.show()

# SECTION 2: Create the CESM case

After generating the MOM6 domain, the next step is to create a CESM case using CrocoDash. This process is straightforward and involves instantiating the CrocoDash Case object. The Case object requires the following inputs:

 - CESM Source Directory: A local path to a compatible CESM source copy. For the workshop, this is the `CESM/` folder of your CROCODILEworkspace install.
 - Case Name: A unique name for the CESM case. For the workshop, cases go in the `croc_cases/` folder of your CROCODILEworkspace install, in a folder named after the case.
 - Input Directory: The directory where all necessary input files will be written. You don't need to put anything in it: CrocoDash creates it and fills it with the grid, bathymetry and forcing files automatically. For the workshop, it's the `croc_input/` folder of your CROCODILEworkspace install.
 - MOM6 Domain Objects: The horizontal grid, topography, and vertical grid created in the previous section.
 - Project ID: (Optional) A project ID, if required by the machine. For the workshop, use `UCGD0009`.
 - Compset: The set of models to be used in the Case. Standalone Ocean, Ocean-BGC, Ocean-Seaice, Ocean-Wave, etc.

If you installed this notebook with CROCODILEworkspace, these paths and the project ID are already filled in for you in the code below.

## Step 2.1: Specify case name and directories:

Begin by specifying the case name and the necessary directory paths. Ensure the CESM root directory points to your own local copy of CESM. Below is an example setup:

In [ ]:
from pathlib import Path

# CESM case (experiment) name
casename = "gs_california_ocn.001"

# CESM source root (Update this path accordingly!!!)
cesmroot = str(Path.cwd().parent / "CESM")  # your workshop install's CESM

# Place where all your input files go 
inputdir = Path.cwd().parent / "croc_input" / casename
    
# CESM case directory
caseroot = Path.cwd().parent / "croc_cases" / casename

## Step 2.2: Create the Case

To create the CESM case, instantiate the `Case` object as shown below. This will automatically set up the CESM case based on the provided inputs: The `cesmroot` argument specifies the path to your local CESM source directory.
The `caseroot` argument defines the directory where the case will be created. CrocoDash will handle all necessary namelist modifications and XML changes to align with the MOM6 domain objects generated earlier.
The `job_queue` and `job_wallclock_time` arguments set the PBS queue and the walltime (`hh:mm:ss`) CESM requests when you submit the run; leave them out to use the machine defaults.

📖 [Case documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/2_case_setup.html)

### Compset quick-reference

The `compset` argument controls which model components are active. Start with the ocean-only default and swap or add stubs as needed:

| What you want | Change in `compset=` |
|---|---|
| **Ocean only (default)** | **`compset="CR_JRA"`** |
| + Waves (WW3) | `compset="CWR_JRA"` |
| + Sea ice (CICE6) | change `CWR_JRA` → `GWR_JRA` |
| + Biogeochemistry (MARBL) | add `%MARBL-BIO` to the MOM6 component, e.g. `MOM6%REGIONAL%MARBL-BIO` |
| + Data runoff (GLOFAS) | use `CWR_JRA_GLOFAS` |
| + Data runoff (JRA) | use `CWR_JRA` with `DROF%JRA` (see runoff notebook) |

For a deeper dive into each option:
- **CICE or BGC**: [Case Setup, Sections 3–4](../case_setup.ipynb#case-cice)
- **Runoff (GLOFAS / JRA / custom)**: [Configure Forcings, Section 5](../configure_forcings.ipynb#forcings-runoff)


In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = 'ncgd0011', # Enter Project ID for PBS Queue
    job_queue = "main", # PBS queue the run is submitted to
    job_wallclock_time = "01:00:00", # hh:mm:ss walltime requested for the run
    override = True,
    machine = "derecho",
    compset = "CR_JRA"
)

In [ ]:
# The data-ice buildnml stops on a "null" ICE_GRID; name it after the case.
# From Casper, a Derecho case needs --non-local.
xml = dict(cwd=caseroot, capture_output=True, text=True, check=True)
query = ["./xmlquery", "ICE_GRID", "--value", "--non-local"]
if subprocess.run(query, **xml).stdout.strip() == "null":
    subprocess.run(["./xmlchange", f"ICE_GRID={casename}", "--non-local"], **xml)

# Section 3: Prepare ocean forcing data

We need to cut out our ocean forcing. The package expects an initial condition and one time-dependent segment per non-land boundary. Naming convention is `"east_unprocessed"` for segments and `"ic_unprocessed"` for the initial condition.

In this notebook, we are forcing with the Copernicus Marine "Glorys" reanalysis dataset. There's a function in the `CrocoDash` package, called `configure_forcings`, that generates a bash script to download the correct boundary forcing files for your experiment. First, you will need to create an account with Copernicus, and then call `copernicusmarine login` to set up your login details on your machine. Then you can run the `get_glorys_data.sh` bash script.

`boundaries` comes from the open-boundary check in Step 1.3.

## Step 3.1 Configure Initial Conditions and Forcings

📖 [Configure forcings docs](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3a_configure_forcings.html) · [Available datasets](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/datasets.html)

In [ ]:
case.configure_forcings(
    date_range = ["2020-03-01 00:00:00", "2020-04-01 00:00:00"],
    boundaries=boundaries,
    product_name = "GLORYS",
    function_name="get_glorys_data_from_rda",
)

```{admonition} Optional forcing additions
:class: tip

You can extend `configure_forcings` with any of these before calling `process_forcings`:

- **Tides**: [Configure Forcings, Section 3](../configure_forcings.ipynb#forcings-tides)
- **Chlorophyll**: [Configure Forcings, Section 4](../configure_forcings.ipynb#forcings-chlorophyll)
- **Alternative data products** (not GLORYS), [Configure Forcings, Section 2](../configure_forcings.ipynb#forcings-data-products)
- **Long runs / offline OBC**: [Process Forcings, Section 2](../process_forcings.ipynb#process-offline)
```

##  Step 3.3: Process forcing data

In this final step, we call the `process_forcings` method of CrocoDash to cut out and interpolate the initial condition as well as all boundaries. CrocoDash also updates MOM6 runtime parameters and CESM xml variables accordingly.

In [ ]:
case.process_forcings()

In [ ]:
print("You can now build and run your case at",caseroot)

# Section 4: Build and run the case

After completing the previous steps, you are ready to build and run your CESM case. Begin by navigating to the case root directory specified during the case creation. Before proceeding, review the `user_nl_mom` file located in the case directory. This file contains MOM6 parameter settings that were automatically generated by CrocoDash. Carefully examine these parameters and make any necessary adjustments to fine-tune the model for your specific requirements. While CrocoDash aims to provide a solid starting point, further tuning and adjustments are typically necessary to improve the model for your use case.

Once you have reviewed and modified the parameters as needed, you can build and execute the case using the following commands. CESM cases must be built and submitted on the machine you passed as `machine` when creating the `Case`. For the workshop, that's **Derecho** (not Casper), so the commands start by logging in to Derecho and going to the case directory:
```
ssh derecho
cd <your workshop install>/croc_cases/gs_california_ocn.001
qcmd -A ncgd0011 -- ./case.build
./case.submit
```

A grid of about 10,000 cells (the default sketch) runs on 48 cores in the `develop` queue: in the case directory, `./xmlchange NTASKS=48,ROOTPE=0` and `./xmlchange JOB_QUEUE=develop,JOB_WALLCLOCK_TIME=01:00:00 --subgroup case.run`.

# Where to go from here?

This tutorial covered the basics of CrocoDash. Extend it with features or jump into a real-world example:

## Add more components
- [Case Setup: Coupling (CICE / BGC)](../case_setup.ipynb#case-cice): sea ice or biogeochemistry
- [Configure Forcings: Tides](../configure_forcings.ipynb#forcings-tides)
- [Configure Forcings: Chlorophyll](../configure_forcings.ipynb#forcings-chlorophyll)
- [Configure Forcings: Runoff](../configure_forcings.ipynb#forcings-runoff)

## Try a different domain
- [Grids: Polar Projections](../grids.ipynb#grids-polar): Arctic & Antarctic domains
- [Grids: Pre-Generated or Subset](../grids.ipynb#grids-pregenerated): pre-generated supergrid or global subset

## Real-world examples
- [NWA12 (North-West Atlantic 1/12°)](../use_cases/three_boundary.ipynb)
- [Antarctic CICE coupled run](../use_cases/mom6_cice_antarctica.ipynb)
- [Low-resolution domain for data assimilation](../use_cases/three_boundary_from_t232.ipynb)

## Explore model output
- [mom6-tools Diagnostics](../../mom6_tools.md)
- [CARIB12 EKE Analysis](../../model2obs/CARIB12_EKE_TS.ipynb)
- [Model–Observation Comparison](../../model2obs/model-obs_comparison.md)

## Full documentation

The complete CrocoDash reference is at **[https://crocodile-cesm.github.io/CrocoDash/latest/](https://crocodile-cesm.github.io/CrocoDash/latest/)**:

- [Installation](https://crocodile-cesm.github.io/CrocoDash/latest/installation.html)
- [Grids (horizontal, topography, vertical)](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/1_grids.html)
- [Available datasets](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/datasets.html)
- [Forcing extraction](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3b_process_forcings.html)
- [Forcing configurations](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3a_configure_forcings.html)
- [Bundling & forking cases](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/shareable.html)
- [API reference](https://crocodile-cesm.github.io/CrocoDash/latest/api-docs/modules.html)